# Chimp tracking - Final deployment pipeline

This notebook deploys the full **enhanced tracking + auto-labelling** pipeline on every video that does not yet have a manual annotation. The goal is to extend the labelled dataset with predictions that are good enough to require only minimal manual correction.

Workflow :

1. **Automated tracking + post-processing**: runs ByteTrack on the body detector, extracts face crops with YOLOX, classifies with ChimpUFE, clusters identities, propagates labels and writes a name-tagged raw output. Unknown chimps are written as `unknown_1`, `unknown_2`, ... (underscore is required: a space breaks the manual annotation parser). A *pre-filled* manual annotation file is also produced so that **most of the correction work consists of fixing wrong predictions, not labelling from scratch**.
2. **Manual corrections → `treated/`**: re-reads the (now possibly edited) annotation file and produces a corrected video with bounding boxes.
3. **Final overlay → `final/`**: same as step 2 but with triangle/name overlays and no frame counter. Only do step 3 when the annotations are 100% correct.

Notes:
- All model weights are loaded **once** at the top of the notebook (loading them per-video would waste ~30 s per video).
- HOTA / recognition evaluation is **not** included here as we are deploying on videos for which we have no ground truth.

## File paths and imports

Specify the input/output directories below. Videos listed in `ignore_S1/S2/S3` are skipped during the corresponding step.

In [1]:
import os, sys
from pathlib import Path

# Resolve notebook directory robustly using __vsc_ipynb_file__ (injected by VS Code)
try:
    FINAL_PIPELINE_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    FINAL_PIPELINE_DIR = Path.cwd().resolve()

UTILS_DIR = FINAL_PIPELINE_DIR / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import pipeline_lib as plib
from pipeline_lib import (
    VFinalPipeline,
    prefill_manual_annotation_file,
    raw_tracking_data_reader, modification_reader, data_writer,
    edit_raw_output, draw_bbox_from_file, mux_audio, has_audio_stream,
    ensure_weights,
)

# ----- weights (local copies in ./weights/) -----
ensure_weights()
WEIGHTS_DIR = FINAL_PIPELINE_DIR / "weights"
BODY_WEIGHTS       = WEIGHTS_DIR / "Body_detection_model.pt"
YOLOX_WEIGHTS      = WEIGHTS_DIR / "yolox_best_only_model.pth"
CLASSIFIER_WEIGHTS = WEIGHTS_DIR / "fine_tune_20.pt"

# ----- video directories -----
PROJECT_ROOT = FINAL_PIPELINE_DIR.parent.parent       # .../ChimpRec
input_video_directory        = str(PROJECT_ROOT / "ChimpVideos" / "input")
output_video_directory       = str(PROJECT_ROOT / "ChimpVideos" / "output")
temp_directory               = f"{output_video_directory}/temp"
raw_text_output_directory    = f"{temp_directory}/raw_output"
manual_annotations_directory = f"{input_video_directory}/manual_annotations"
treated_directory            = f"{output_video_directory}/treated"
final_directory              = f"{output_video_directory}/final"

for path in [
    input_video_directory, output_video_directory, temp_directory,
    raw_text_output_directory, manual_annotations_directory,
    treated_directory, final_directory,
]:
    os.makedirs(path, exist_ok=True)

# Videos to ignore per step
ignore_S1 = [
    "20241019 - 14h29",
    "short"
]
ignore_S2 = []
ignore_S3 = []

# Step 1 skips a video when its manual annotation file already exists *and is non-empty*
# (i.e. it has already been pre-filled or hand-edited). Toggle this off to re-run.
SKIP_VIDEOS_WITH_EXISTING_ANNOTATION = False

print(f"pipeline dir : {FINAL_PIPELINE_DIR}")
print(f"input  : {input_video_directory}")
print(f"output : {output_video_directory}")
print(f"manual : {manual_annotations_directory}")

[weights] OK  /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/Body_detection_model.pt
[weights] OK  /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/yolox_best_only_model.pth
[weights] OK  /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/fine_tune_20.pt
pipeline dir : /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline
input  : /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/input
output : /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/output
manual : /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/input/manual_annotations


In [2]:
# Load all models once (vFinal) (body detector, YOLOX face detector, ChimpUFE classifier).
# This cell can take ~30 s the first time.
pipeline = VFinalPipeline(
    body_weights       = BODY_WEIGHTS,
    yolox_weights      = YOLOX_WEIGHTS,
    classifier_weights = CLASSIFIER_WEIGHTS,
)

[VFinalPipeline] device = cuda
[VFinalPipeline] body  weights = /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/Body_detection_model.pt
[VFinalPipeline] yolox weights = /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/yolox_best_only_model.pth
[VFinalPipeline] clf   weights = /home/diego/Desktop/ChimpRec/ChimpRec/Code/FinalPipeline/weights/fine_tune_20.pt
  ChimpUFE fine-tuned (stage=stage2_full_finetune, val_top1=0.889)
[VFinalPipeline] classes (20): ['Amadi', 'Banalia', 'Binasera', 'Djiku', 'Ivan', 'Jeje', 'Kalemi', 'Kassongo', 'Kira', 'Lwama', 'Malago', 'Maniema', 'Mazingira', 'Muke', 'Nganja', 'Nzuri', 'Penda', 'Talisa', 'Tanganica', 'Tingitingi']


## Step 1 — Automated tracking + auto-labelling

For every video in `input_video_directory` that does not yet have a (non-empty) manual annotation file:

1. Run the vFinal pipeline (ByteTrack → face crops → classification → clustering → propagation → co-alive duplicate-name resolution → short-gap interpolation).
2. Write the raw per-frame tracking output to `temp/raw_output/{video_name}.txt`.
3. Pre-fill the manual annotation file at `input/manual_annotations/{video_name}.txt` with the predicted identities. **The user only needs to correct the mistakes.**
4. Render an MP4 preview with bounding boxes + names into `temp/{video_name}-(temp)-audio.mp4`.

In [3]:
# ---------- STEP 1: enhanced tracking + auto-labelling ----------
for input_video in sorted(os.listdir(input_video_directory)):
    if not (input_video.lower().endswith(".mp4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored (in ignore_S1)")
        continue

    annotation_file_path = f"{manual_annotations_directory}/{video_name}.txt"
    if SKIP_VIDEOS_WITH_EXISTING_ANNOTATION and os.path.exists(annotation_file_path) \
            and os.path.getsize(annotation_file_path) > 0:
        print(f"{video_name}.txt already present in {manual_annotations_directory}; skipping.")
        continue
    print(f"\n--- Processing {video_name} (vFinal pipeline) ---")

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"

    # 1+2+3. Run the vFinal pipeline and write the raw output txt.
    id_map = pipeline.process(
        video_path      = full_video_path,
        output_txt_path = raw_txt_path,
    )

    # 3. Pre-fill the manual annotation file with the predicted identities.
    prefill_manual_annotation_file(id_map, annotation_file_path)
    print(f"  pre-filled manual annotation file: {annotation_file_path}")

    # 4. Render preview MP4 (bbox + name + frame counter).
    preview_path = f"{temp_directory}/{video_name}-(temp)-audio.mp4"
    draw_bbox_from_file(
        file_path        = raw_txt_path,
        input_video_path = full_video_path,
        output_video_path= preview_path,
        annotation_type  = "bbox",
        draw_frame_count = True,
    )
    if has_audio_stream(full_video_path):
        print("  adding audio...")
        mux_audio(full_video_path, preview_path, preview_path)
    else:
        print("  no audio stream detected; skipping mux.")
    print(f"  preview: {preview_path}")


--- Processing 20241104 - 11h33 (vFinal pipeline) ---

[VFinalPipeline] >>> 20241104 - 11h33.MP4
  Detecting on 46344 frames @ 50.0 fps ...


ByteTrack: 100%|██████████| 46344/46344 [00:23<00:00, 1982.75it/s]


  Sampling 16142 unique frames for 522 tracks


Classify+Emb: 100%|██████████| 522/522 [01:05<00:00,  7.97it/s]    


  Candidate edges (sim >= 0.55): 620
  Merges accepted: 154  (rejected: overlap=231, label-conflict=4)
  Cross-cluster name-merges: accepted=0, rejected=6
  Cluster-aggregated classification: 55 clusters labelled (of 186 total)
  Cross-cluster name-merges: accepted=24, rejected=13
  Embedding propagation (sim >= 0.5): accepted=10, overlap-rejected=6
  -> labelled=185  unknown=180  dropped=157  identities=19
     Kira          : 19 tracks
     Penda         : 19 tracks
     Binasera      : 19 tracks
     Malago        : 16 tracks
     Maniema       : 16 tracks
     Nganja        : 13 tracks
     Djiku         : 12 tracks
     Lwama         : 10 tracks
     Tanganyika    : 9 tracks
     Kassongo      : 9 tracks
     Amadi         : 8 tracks
     Banalia       : 8 tracks
     Ivan          : 7 tracks
     Jeje          : 6 tracks
     Muki          : 5 tracks
     Talissa       : 4 tracks
     Kalemi        : 2 tracks
     Nzuri         : 2 tracks
     Mazingara     : 1 tracks
  [write] 1

Drawing annotations (20241104 - 11h33.MP4): 100%|██████████| 46344/46344 [19:12<00:00, 40.23it/s]


  adding audio...


ffmpeg version n8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 16.1.1 (GCC) 20260430
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-lcms2 --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-l

  preview: /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/output/temp/20241104 - 11h33-(temp)-audio.mp4
short.mp4 ignored (in ignore_S1)


[out#0/mp4 @ 0x55939ce5c600] video:6933753KiB audio:173790KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.013815%
frame=46344 fps=6166 q=-1.0 Lsize= 7108525KiB time=00:15:26.88 bitrate=62826.9kbits/s speed= 123x elapsed=0:00:07.51    


## Step 2 — Apply manual edits → `treated/`

Edit the files in `input/manual_annotations/` (one per video). The format is one identity per line:

```
Amadi: Amadi unknown_3
Maniema: Maniema
unknown_1: unknown_1
```

Use `Name: token1 token2 ...` to group several auto-predicted ids into a single chimp, and `Name: token*start-end` to scope a re-label to a frame range. Use `SWAP: frame id_a id_b` to swap two ids from a given frame onward. See `Code/Tracking/Bytetrack/ui_lib_bytetrack.py` for the full syntax.

In [ ]:
# ---------- STEP 2: apply manual edits -> treated/ ----------
for input_video in sorted(os.listdir(input_video_directory)):
    if not (input_video.lower().endswith(".mp4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored (in ignore_S2)")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_txt_path    = f"{raw_text_output_directory}/{video_name}.txt"
    if not os.path.exists(raw_txt_path):
        print(f"{video_name}: no raw output yet ({raw_txt_path}); run step 1 first.")
        continue

    raw_reader = raw_tracking_data_reader(raw_txt_path)
    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(f"Error: manual annotation file not found for <{full_video_path}> at <{annotation_file}>.")
        continue

    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)
    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path  = os.path.join(video_out_dir, f"{video_name}-treated.mp4")

    writer = data_writer(metadata_file_path)
    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path        = metadata_file_path,
        input_video_path = full_video_path,
        output_video_path= output_video_path,
        annotation_type  = "bbox",
        draw_frame_count = True,
    )
    if has_audio_stream(full_video_path):
        print("  adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("  no audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}\n")

## Step 3 — Final overlay (`final/`)

Same as Step 2 but renders triangle+name overlays (publication-style). **Only run this when the manual annotation file is 100 % correct.**

In [ ]:
# ---------- STEP 3: final arrows/names -> final/ ----------
for input_video in sorted(os.listdir(input_video_directory)):
    if not (input_video.lower().endswith(".mp4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored (in ignore_S3)")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_txt_path    = f"{raw_text_output_directory}/{video_name}.txt"
    if not os.path.exists(raw_txt_path):
        print(f"{video_name}: no raw output yet ({raw_txt_path}); run step 1 first.")
        continue

    raw_reader = raw_tracking_data_reader(raw_txt_path)
    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(f"Error: manual annotation file not found for <{full_video_path}> at <{annotation_file}>.")
        continue

    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)
    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path  = os.path.join(video_out_dir, f"{video_name}-final.mp4")

    writer = data_writer(metadata_file_path)
    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path        = metadata_file_path,
        input_video_path = full_video_path,
        output_video_path= output_video_path,
        annotation_type  = "triangle",
        draw_frame_count = False,
    )
    if has_audio_stream(full_video_path):
        print("  adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("  no audio stream detected; skipping mux.")
    print(f"Final render done: {full_video_path}\n")